# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anasYaha/Flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: scoring / ranking, built on top of a classification sub-model.**

My lane (Refresh / Content Opportunity Scoring) does not need a single yes/no label as its final output — it needs an ordered list, because a reviewer only has capacity to check the top N pages, not every page tagged "positive." So the outer task is **scoring/ranking**: every page gets a continuous priority score, and pages are sorted by it.

Under the hood, that score is built from a **classification sub-model** (probability that a page is declining / needs review), combined with a transparent rule-based baseline score. This matches how the starter pipeline itself is built: a classifier produces `best_model_probability`, which is then blended with `normalized_baseline_score` into the final ranking score. I'm not doing clustering (I'm not looking for groups of similar pages) and not doing plain classification alone (a bare yes/no per page ignores that reviewer capacity is limited and some "positives" are more urgent than others).

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Starting proxy (what I'll use early on):**

```
is_declining_label = trend_direction == "down"
```

This is the same proxy the starter pipeline uses. It comes from a **defined rule** applied to the *current* observation window, not from a future outcome — that is its main weakness. It tells me a page looks like it's declining right now, not whether it will keep declining, recover on its own, or get worse. I'm using it as a starting point because it's simple, already computed in the starter data, and lets me get a first honest baseline-vs-model comparison running quickly.

**Stronger target I'm aiming for by the capstone (an observed future outcome, not a rule):**

```
features from the prior 90 days -> decline (or recovery) observed over the next 30 days
```

This future-window label comes from an **observed outcome** rather than a same-window rule, which avoids the proxy's core weakness and requires a proper leakage audit (feature window must never overlap the target window) once I move to the full warehouse daily-facts table.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.**

Of the top 50 pages my ranking puts at the top, how many are actually true positives (genuinely declining / worth reviewing) by my label? I'm choosing this over plain accuracy or ROC AUC because it matches how the output is actually used: a reviewer opens a fixed-size list, not the whole dataset, so what matters is whether the *top* of the list is trustworthy — not how the model performs across every page, most of which no one will ever look at.

**What "good" looks like:** the starter pipeline's own results give me a concrete bar to beat — the fixed baseline rule scored Precision@50 = 0.240 (about 12 of the top 50 correct), while a random forest model reached 0.740 (about 37 of 50 correct) on the starter slice. So for my own work, "good" means clearly beating the transparent baseline rule's precision@50, not just beating a coin flip.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd

csv_url = "https://raw.githubusercontent.com/anasYaha/Flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_url)

# Same filters as the starter pipeline: visible, mature pages only
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates(subset="content_id")

# Sketch the starting proxy target
df["is_declining_label"] = df["trend_direction"] == "down"

key_cols = [
    "content_id", "client_id", "impressions_90d", "sessions_90d",
    "content_age_days", "days_since_last_update", "avg_position",
    "ctr", "trend_direction", "is_declining_label",
]
key_cols = [c for c in key_cols if c in df.columns]

print(f"Rows (= pages) in this slice: {len(df)}")
print(f"Positive rate for the proxy label: {df['is_declining_label'].mean() * 100:.1f}%")
df[key_cols].head()

Rows (= pages) in this slice: 30000
Positive rate for the proxy label: 54.2%


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,days_since_last_update,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,17,187,20,10.6,0.76,down,True
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,25,20.3,0.05,down,True
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,20,36.5,0.09,down,True
3,content_331d6c4de07b,client_19581e27de,11751,78,463,22,6.2,0.49,stable,False
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,14,44.0,0.13,down,True


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed if-statement rule can only combine a handful of thresholds in a way a human picked by hand (e.g. "stale AND visible" or "declining AND has demand"). That works as a first pass, but it can't capture how many signals interact *together and non-linearly* — for example, a moderate CTR drop might only matter when it's paired with a specific position range and a specific content age, a combination no one would think to hand-write as a rule.

This isn't just a theoretical argument — the starter pipeline already measured it: the fixed baseline rule reached ROC AUC 0.627 and Precision@50 0.240, while a random forest trained on the same signals reached ROC AUC 0.750 and Precision@50 0.740 — roughly three times as many correct pages in the top 50. That gap is real evidence that the relationships between these signals are too messy for a small set of hand-written thresholds to capture, and that a model which can learn interactions between signals finds real structure a fixed rule misses.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.